In [16]:
# ============================================================
# 01_build_network.ipynb
# BUILD AND SAVE AIRPORT COMPLEX NETWORK
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from collections import Counter

import community as community_louvain

import pickle
from pathlib import Path

plt.style.use("ggplot")

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "../data"

print("DATA DIRECTORY:")
print(DATA_DIR)

# ============================================================
# LOAD DATASETS
# ============================================================

print("\n" + "=" * 60)
print("LOADING DATASETS")
print("=" * 60)

# ------------------------------------------------------------
# AIRPORTS
# ------------------------------------------------------------

airport_cols = [
    "airport_id",
    "name",
    "city",
    "country",
    "iata",
    "icao",
    "latitude",
    "longitude",
    "altitude",
    "timezone",
    "dst",
    "tz_database",
    "type",
    "source"
]

airports = pd.read_csv(
    DATA_DIR / "airports.dat",
    header=None,
    names=airport_cols
)

print("\nAirports loaded:")
print(airports.shape)

# ------------------------------------------------------------
# ROUTES
# ------------------------------------------------------------

route_cols = [
    "airline",
    "airline_id",
    "source",
    "source_id",
    "destination",
    "destination_id",
    "codeshare",
    "stops",
    "equipment"
]

routes = pd.read_csv(
    DATA_DIR / "routes.dat",
    header=None,
    names=route_cols
)

print("\nRoutes loaded:")
print(routes.shape)

# ============================================================
# CLEAN DATA
# ============================================================

print("\n" + "=" * 60)
print("CLEANING DATA")
print("=" * 60)

# Remove invalid airports
airports = airports[
    airports["iata"] != "\\N"
]

# Remove invalid routes
routes = routes[
    (routes["source"] != "\\N") &
    (routes["destination"] != "\\N")
]

print("Valid airports:", len(airports))
print("Valid routes:", len(routes))

# ============================================================
# CREATE GRAPH
# ============================================================

print("\n" + "=" * 60)
print("CREATING GRAPH")
print("=" * 60)

G = nx.from_pandas_edgelist(
    routes,
    source="source",
    target="destination",
    create_using=nx.Graph()
)

print("Initial graph created")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# ============================================================
# EXTRACT GIANT COMPONENT
# ============================================================

print("\nExtracting giant connected component...")

largest_cc = max(
    nx.connected_components(G),
    key=len
)

G_main = G.subgraph(largest_cc).copy()

print("Giant component extracted")

print("Nodes:", G_main.number_of_nodes())
print("Edges:", G_main.number_of_edges())

# ============================================================
# ADD AIRPORT ATTRIBUTES
# ============================================================

print("\n" + "=" * 60)
print("ADDING NODE ATTRIBUTES")
print("=" * 60)

airport_dict = airports.set_index("iata").to_dict("index")

for node in G_main.nodes():

    if node in airport_dict:

        for key, value in airport_dict[node].items():

            G_main.nodes[node][key] = value

print("Airport attributes added")

# ============================================================
# BASIC NETWORK METRICS
# ============================================================

print("\n" + "=" * 60)
print("COMPUTING NETWORK METRICS")
print("=" * 60)

# ------------------------------------------------------------
# Degree
# ------------------------------------------------------------

degrees = dict(G_main.degree())

nx.set_node_attributes(
    G_main,
    degrees,
    "degree"
)

avg_degree = np.mean(list(degrees.values()))

print("Average degree:", avg_degree)

# ------------------------------------------------------------
# Clustering
# ------------------------------------------------------------

clustering = nx.clustering(G_main)

nx.set_node_attributes(
    G_main,
    clustering,
    "clustering"
)

avg_clustering = nx.average_clustering(G_main)

print("Average clustering:", avg_clustering)

# ------------------------------------------------------------
# Betweenness
# ------------------------------------------------------------

print("\nComputing betweenness centrality...")
print("(This may take several minutes)")

betweenness = nx.betweenness_centrality(
    G_main,
    k=500,
    seed=42
)

nx.set_node_attributes(
    G_main,
    betweenness,
    "betweenness"
)

print("Betweenness completed")

# ------------------------------------------------------------
# Closeness
# ------------------------------------------------------------

print("\nComputing closeness centrality...")

closeness = nx.closeness_centrality(G_main)

nx.set_node_attributes(
    G_main,
    closeness,
    "closeness"
)

print("Closeness completed")

# ============================================================
# COMMUNITY DETECTION
# ============================================================

print("\n" + "=" * 60)
print("COMMUNITY DETECTION")
print("=" * 60)

partition = community_louvain.best_partition(G_main)

nx.set_node_attributes(
    G_main,
    partition,
    "community"
)

num_communities = len(
    set(partition.values())
)

print("Number of communities:", num_communities)

community_sizes = Counter(
    partition.values()
)

print("\nLargest communities:")

print(
    community_sizes.most_common(10)
)

# ============================================================
# GLOBAL METRICS
# ============================================================

print("\n" + "=" * 60)
print("GLOBAL METRICS")
print("=" * 60)

density = nx.density(G_main)

print("Density:", density)

avg_path = nx.average_shortest_path_length(
    G_main
)

print("Average shortest path length:", avg_path)

diameter = nx.diameter(G_main)

print("Diameter:", diameter)

# ============================================================
# SAVE GRAPH
# ============================================================

print("\n" + "=" * 60)
print("SAVING NETWORK")
print("=" * 60)

# ------------------------------------------------------------
# Save as GPickle
# ------------------------------------------------------------

# ------------------------------------------------------------
# Save graph with pickle
# ------------------------------------------------------------

gpickle_path = DATA_DIR / "airport_network.gpickle"

with open(gpickle_path, "wb") as f:
    pickle.dump(G_main, f)

print("Saved:")
print(gpickle_path)
# ------------------------------------------------------------
# Save as GraphML
# ------------------------------------------------------------

graphml_path = DATA_DIR / "airport_network.graphml"

nx.write_graphml(
    G_main,
    graphml_path
)

print("\nSaved:")
print(graphml_path)

# ============================================================
# SAVE EXTRA OBJECTS
# ============================================================

print("\nSaving auxiliary objects...")

# ------------------------------------------------------------
# Louvain partition
# ------------------------------------------------------------

with open(
    DATA_DIR / "louvain_partition.pkl",
    "wb"
) as f:

    pickle.dump(partition, f)

# ------------------------------------------------------------
# Degree distribution
# ------------------------------------------------------------

with open(
    DATA_DIR / "degrees.pkl",
    "wb"
) as f:

    pickle.dump(degrees, f)

# ------------------------------------------------------------
# Betweenness
# ------------------------------------------------------------

with open(
    DATA_DIR / "betweenness.pkl",
    "wb"
) as f:

    pickle.dump(betweenness, f)

print("Auxiliary files saved")

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print("""
Airport network successfully built and saved.

Generated files:

- airport_network.gpickle
- airport_network.graphml
- louvain_partition.pkl
- degrees.pkl
- betweenness.pkl

The graph now contains:

- airport metadata
- geographical coordinates
- centrality metrics
- community labels
- clustering coefficients

Future notebooks can directly load
the processed graph without recomputing.
""")

DATA DIRECTORY:
/home/mzhc13/master/complex-networks-airports/notebooks/../data

LOADING DATASETS

Airports loaded:
(7698, 14)

Routes loaded:
(67663, 9)

CLEANING DATA
Valid airports: 6072
Valid routes: 67663

CREATING GRAPH
Initial graph created
Nodes: 3425
Edges: 19257

Extracting giant connected component...
Giant component extracted
Nodes: 3397
Edges: 19231

ADDING NODE ATTRIBUTES
Airport attributes added

COMPUTING NETWORK METRICS
Average degree: 11.322343244038859
Average clustering: 0.48833620245296055

Computing betweenness centrality...
(This may take several minutes)
Betweenness completed

Computing closeness centrality...
Closeness completed

COMMUNITY DETECTION
Number of communities: 21

Largest communities:
[(6, 703), (2, 597), (3, 531), (5, 401), (1, 276), (9, 246), (10, 183), (0, 169), (8, 76), (7, 52)]

GLOBAL METRICS
Density: 0.003334023334522632
Average shortest path length: 4.103241167898093
Diameter: 13

SAVING NETWORK
Saved:
/home/mzhc13/master/complex-networks-ai